# Metrics: balanced outcomes and game closeness

**Milestone:** "Balanced outcomes: define the metric (for example, the share of games drawn or
decided narrowly) and a target over a fixed number of games. Initial aspiration: 95 to 99 percent"
(README Teaching). This notebook makes the definition concrete so it can be argued with.

**Proposed definition** (all thresholds in game.yaml `teaching`):

* A game is **balanced** if it is a draw, or it is decisive **narrowly**: the engine eval (White
  POV, after each ply) stayed within `+-narrow_decisive_cp` for every ply except the last
  `late_plies` (default 10), i.e. nobody was clearly winning until the very end.
* **Balanced share** over N games, with a Wilson 95% interval (with N = 20 and 19 balanced
  games, the interval is roughly 0.76..0.99: say so when reporting).
* Supporting statistics: eval swing (max - min), mean |delta eval| per ply, number of lead changes,
  and a per-game closeness in 0..1.

**TODO checklist**
- [ ] Wilson interval.
- [ ] Per-game metric functions (draw, narrow decisive, balanced, swing stats, closeness).
- [ ] Over-N-games summaries (`balanced_share`, `summarize_games`).
- [ ] Self-checks (plan at the bottom).
- [ ] Decide `late_plies` and `narrow_decisive_cp` with real games; maybe use win probability
      instead of centipawns.
- [ ] Fix N (number of games) and the target before user testing (README).

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import yaml

with open(Path("../../gambit_bringup/config/game.yaml")) as _f:
    METRIC_CFG = yaml.safe_load(_f)["teaching"]

## Wilson score interval (for any share k/n)

In [ ]:
def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """(low, high) Wilson score interval for k successes out of n; (0, 1) when n == 0."""
    # Design: the Wilson interval (not the normal "p +- z*sqrt(p(1-p)/n)" one) behaves well for
    # shares near 1 and small n, which is exactly the 95-99 % target with ~20 games.
    # Formula: centre = (p + z^2/2n) / (1 + z^2/n),
    #          half   = z * sqrt(p(1-p)/n + z^2/4n^2) / (1 + z^2/n); clip to 0..1.
    # TODO: implement
    raise NotImplementedError

## Per-game metrics

In [ ]:
@dataclass
class GameOutcome:
    result: str                                   # "1-0" | "0-1" | "1/2-1/2"
    evals_cp: list = field(default_factory=list)  # White POV after each ply
    robot_color: str = "black"
    game_id: str = ""


def is_draw(g: GameOutcome) -> bool:
    # TODO: implement
    raise NotImplementedError


def is_narrow_decisive(g: GameOutcome, narrow_cp: int = METRIC_CFG["narrow_decisive_cp"],
                       late_plies: int = 10) -> bool:
    """Decisive, and |eval| stayed within narrow_cp for every ply except the last `late_plies`."""
    # Design: only decisive results qualify; ignore the final late_plies (the winner converting
    # at the end is expected); a game shorter than late_plies counts as narrow.
    # TODO: implement
    raise NotImplementedError


def is_balanced(g: GameOutcome, **kw) -> bool:
    """Draw or narrowly decisive."""
    # TODO: implement
    raise NotImplementedError


def eval_swing_stats(evals_cp: list, lead_cp: int = 50, cap: int = 1000) -> dict:
    """{max_abs, swing, mean_abs_delta, lead_changes} for one game's eval curve."""
    # Design:
    # - Clip evals to +-cap first (mate scores would swamp every statistic).
    # - swing = max - min; mean_abs_delta = mean of |np.diff|.
    # - lead_changes: count sign changes among plies where someone leads by at least lead_cp
    #   (small wobbles around 0 are not lead changes).
    # - Empty list -> all zeros.
    # TODO: implement
    raise NotImplementedError


def closeness(evals_cp: list, band: int = METRIC_CFG["closeness_band_cp"] * 4) -> float:
    """1 = eval always 0, 0 = always beyond +-band. Mean of 1 - min(|e|, band)/band."""
    # TODO: implement (empty list -> 1.0)
    raise NotImplementedError

## Over N games

In [ ]:
def balanced_share(games: list[GameOutcome], **kw) -> dict:
    """{n, balanced, share, ci_low, ci_high, meets_target} over a list of games."""
    # Design: count balanced games, Wilson interval on the share; meets_target compares the
    # *lower* bound with METRIC_CFG["balanced_target"] (a claim should survive the uncertainty).
    # TODO: implement
    raise NotImplementedError


def summarize_games(games: list[GameOutcome]) -> pd.DataFrame:
    """One row per game with every per-game metric above."""
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - wilson_interval(19, 20) is about (0.76, 0.99..1.0); wilson_interval(0, 0) == (0.0, 1.0).
# - A close win (small evals, then a late conversion over the last 10 plies) is narrow decisive;
#   a blowout (+400 early) is not; a draw is balanced regardless of its evals.
# - balanced_share([close_win, blowout, draw]): 2 balanced, share 2/3, does not meet the target.
# - eval_swing_stats([0, 100, -100, 100]): 2 lead changes, swing 200, max_abs 100.
# - closeness([0, 0]) == 1.0, closeness([10_000]) == 0.0; summarize_games returns one row per game.
# Example shape of a test:
# assert wilson_interval(0, 0) == (0.0, 1.0)
print("Self-check: not written yet (see plan above)")